# Task 6 · Inference and scoring only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Install GPU environment
Restart the runtime if requested, then rerun clone/setup. GPU training has not been run on the CPU-only development machine.


In [ ]:
%pip -q install -r requirements.txt
!python -m pip check
!nvidia-smi


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = ''  # REQUIRED: ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = ''  # auto-detect when the config contains one Google Drive remote


In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y rclone
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    subprocess.run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
subprocess.run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Prepare data on Colab SSD
The download location is automatically derived from ROOT_FOLDER_ID.


In [ ]:
USE_DRIVE_DATA = True
if USE_DRIVE_DATA:
    copy(cloud('task6-data/raw'), 'data/raw')
else:
    subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## Experiment settings
Only ROOT_FOLDER_ID determines cloud storage. RUN_NAME groups your experiments under it.
Periodic checkpoints upload after each save; final adapter uploads after training finishes.
Local SSD is used for training and inference. A disconnected runtime can only recover checkpoints already uploaded.


In [ ]:
import json
USE_4BIT = True
RUN_FULL = False
RUN_NAME = 'experiment01'
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME must contain only letters, numbers, underscores or hyphens')
RUNS = Path('outputs/runs') / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT_STAGE = 'sft' if RUN_FULL else 'sft-smoke'
GRPO_STAGE = 'grpo' if RUN_FULL else 'grpo-smoke'
SFT = str(RUNS / SFT_STAGE)
GRPO = str(RUNS / GRPO_STAGE)
common = {'model.load_in_4bit': USE_4BIT}

def run(module, config, overrides, local_output, cloud_subdir):
    cmd = [sys.executable, '-m', module, '--config', config]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={json.dumps(value)}']
    env = dict(os.environ)
    env.update(MMCQA_RCLONE_CONFIG=str(config_path), MMCQA_DRIVE_ROOT_ID=root_id,
               MMCQA_SYNC_DESTINATION=cloud(cloud_subdir))
    result = subprocess.run(cmd, env=env)
    # Also preserve final adapter and logs, including after a failed training subprocess.
    if Path(local_output).exists():
        copy(local_output, cloud(cloud_subdir))
    result.check_returncode()


## Test inference only
Select run/stage names below. No manual Drive filesystem paths are needed.
Validation scoring belongs to the SFT/GRPO notebooks. This notebook accepts test splits only.
Current devtest is unlabelled: generate predictions to submit; no local score can be computed.


In [ ]:
INPUT_RUN_NAME = RUN_NAME
SPLIT = 'devtest'
TEST_LIMIT = None  # full split; optionally 4 for an inference smoke test
if SPLIT not in {'devtest', 'test'}:
    raise ValueError('Use devtest/test here; validation is evaluated in the training notebooks')
ADAPTER_STAGES = {'sft': SFT_STAGE, 'grpo': GRPO_STAGE}
predictions = RUNS / 'predictions'
for label, stage in ADAPTER_STAGES.items():
    adapter = Path('outputs/inputs/inference') / INPUT_RUN_NAME / stage / 'final'
    copy(cloud(f'task6-runs/{INPUT_RUN_NAME}/{stage}/final'), adapter)
    overrides = {**common, 'model.adapter': str(adapter), 'model.adapter_revision': None,
                 'inference.split': SPLIT, 'inference.output': str(predictions / f'{label}_{SPLIT}.jsonl')}
    overrides['data.limit'] = TEST_LIMIT
    run('mmcultureqa.inference.predict', 'configs/inference.yaml', overrides,
        predictions, f'task6-runs/{RUN_NAME}/predictions')


In [ ]:
from mmcultureqa.common import read_jsonl, write_jsonl
test_rows = read_jsonl(Path('data/prepared') / f'{SPLIT}.jsonl')
if TEST_LIMIT is not None:
    test_rows = test_rows[:TEST_LIMIT]
if test_rows and all(row.get('answer') for row in test_rows):
    gold = predictions / f'{SPLIT}_gold_en.jsonl'
    write_jsonl(gold, test_rows)
    for label in ADAPTER_STAGES:
        subprocess.run([sys.executable, 'vendor/organizer/scorer/score.py',
                        '--gold', str(gold), '--lang', 'en',
                        '--pred', str(predictions / f'{label}_{SPLIT}.jsonl'),
                        '--out', str(predictions / f'{label}_{SPLIT}_scores.json')], check=True)
    copy(predictions, cloud(f'task6-runs/{RUN_NAME}/predictions'))
else:
    print(f'{SPLIT} has no public answers. Predictions saved for submission; no local score available.')
